In [1]:
!pip install -U flax optax icecream jax[cuda12] jaxlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.3/531.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 MB 34.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 189.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 110.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.8/175.8 MB 16.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 180.0 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: jax-cuda12-pjrt
    Found existing installation: jax-cuda12-pjrt 0.7.2
    Uninstalling jax-cuda12-pjrt-0.7.2:
      Successfully uninstalled jax-cuda12-pjrt-0.7.2
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: jax-cuda12-plugin
    Found existing installation: jax-cuda12-plugin 0.7.2
    Uninstalling jax-cuda12-plugin-0.7.2:
     

In [2]:
!git clone https://github.com/RaameshB/JAX-Mambas.git

Cloning into 'JAX-Mambas'...
remote: Enumerating objects: 400, done.
remote: Counting objects: 100% (228/228), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 400 (delta 117), reused 172 (delta 66), pack-reused 172 (from 1)
Receiving objects: 100% (400/400), 323.41 KiB | 1.66 MiB/s, done.
Resolving deltas: 100% (215/215), done.


In [3]:
!cd JAX-Mambas && git switch cuda-selective-scan && git pull

Branch 'cuda-selective-scan' set up to track remote branch 'cuda-selective-scan' from 'origin'.
Switched to a new branch 'cuda-selective-scan'
Already up to date.


In [2]:
import sys
sys.path.append("/content/JAX-Mambas")

In [3]:
import jax
from jax import numpy as jnp
from jax import random
from functools import partial
from flax import nnx
from mamba1 import Mamba
import optax
from icecream import ic

In [4]:
def generate_induction_heads(rng_key, seq_len=256, vocab_size=16):
    special_key, content_key = random.split(rng_key)
    special_token = jnp.array([vocab_size-1])
    sequence = jnp.concat((random.randint(content_key, (seq_len-1,), minval=0, maxval=vocab_size-1), special_token))
    special_loc = random.randint(special_key, (1,), minval=0, maxval=seq_len-2)
    sequence_with_key = sequence.at[special_loc[...]].set(jnp.array(special_token))
    value = sequence_with_key[special_loc[...]+1]
    return sequence_with_key, value[0]


def create_batch(key, bsz, seq_len=256, vocab_size=16):
    induction_heads_batch, values = jax.vmap(
        partial(generate_induction_heads, seq_len=seq_len, vocab_size=vocab_size)
    )(random.split(key, bsz))
    one_hot_y = jax.nn.one_hot(values, vocab_size)
    return induction_heads_batch, one_hot_y

In [5]:
class MambaInductionHeads(nnx.Module):
    def __init__(self, rngs, vocab_size=16, D=64, expand=2, num_layers=2):
        self.embed = nnx.Embed(num_embeddings=vocab_size, features=D, rngs=rngs)
        self.mambas = nnx.List([Mamba(D=D, expand=expand, rngs=rngs) for _ in range(num_layers)])
        self.proj_down = nnx.Linear(in_features=D, out_features=vocab_size, rngs=rngs)
    def __call__(self, x):
        hidden = self.embed(x)
        for mamba in self.mambas:
            hidden = mamba(hidden)
        return self.proj_down(hidden)

In [41]:
lr = 1e-4
bsz = 256
train_steps = 10000

In [42]:
rngs = nnx.Rngs(0)
model = MambaInductionHeads(rngs=rngs)
graphdef, params = nnx.split(model, nnx.Param)
optimizer = optax.contrib.cocob()
opt_state = optimizer.init(params)

In [43]:
@nnx.jit
def train_step(rngs, graphdef, params, opt_state):
    def compute_loss(params, inputs, labels):
        model = nnx.merge(graphdef, params)
        logits = model(inputs)[:,-1]
        loss = jnp.mean(optax.losses.safe_softmax_cross_entropy(logits, labels))
        return loss
    batch_x, batch_y = create_batch(rngs.inputs(), bsz=bsz)
    loss, grads = jax.value_and_grad(compute_loss)(params, batch_x, batch_y)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

In [45]:
for step in range(train_steps):
    params, opt_state, loss = train_step(rngs, graphdef, params, opt_state)

    print(f'Step: {step}, Loss: {loss}')

Step: 0, Loss: 2.5339095373055898e-05
Step: 1, Loss: 4.2324376408942044e-05
Step: 2, Loss: 2.5084964363486506e-05
Step: 3, Loss: 5.819090802106075e-05
Step: 4, Loss: 2.6433250241097994e-05
Step: 5, Loss: 3.4799452350853244e-06
Step: 6, Loss: 2.0995117665734142e-05
Step: 7, Loss: 2.4465101887471974e-05
Step: 8, Loss: 1.4944881513656583e-05
Step: 9, Loss: 3.0768966098548844e-05
Step: 10, Loss: 5.1631828682729974e-05
Step: 11, Loss: 0.00014690225361846387
Step: 12, Loss: 2.8434948035283014e-05
Step: 13, Loss: 0.00011756575986510143
Step: 14, Loss: 8.801200601737946e-05
Step: 15, Loss: 1.3061057870800141e-05
Step: 16, Loss: 7.189420284703374e-05
Step: 17, Loss: 1.667063042987138e-05
Step: 18, Loss: 2.3143387807067484e-05
Step: 19, Loss: 2.325759305676911e-05
Step: 20, Loss: 2.527262768126093e-05
Step: 21, Loss: 2.9473892936948687e-05
Step: 22, Loss: 0.00011372988956281915
Step: 23, Loss: 2.5309542252216488e-05
Step: 24, Loss: 1.5939689546939917e-05
Step: 25, Loss: 2.5143210223177448e-05
St

In [46]:
model = nnx.merge(graphdef, params)

In [47]:
num_samples = 128
batch, labels = create_batch(rngs.inputs(), bsz=num_samples)
logits = jnp.argmax(model(batch)[:,-1], axis=-1)
reference = jnp.argmax(labels, axis=-1)
correct_proportion = jnp.sum(logits==reference)/num_samples
correct_proportion

Array(1., dtype=float32)

In [48]:
num_samples = 32
batch, labels = create_batch(rngs.inputs(), bsz=num_samples, seq_len=2**17)
logits = jnp.argmax(model(batch)[:,-1], axis=-1)
reference = jnp.argmax(labels, axis=-1)
correct_proportion = jnp.sum(logits==reference)/num_samples
correct_proportion

Array(0.0625, dtype=float32)

In [49]:
1/16

0.0625

In [42]:
batch, labels = create_batch(rngs.inputs(), bsz=1)
outs = model(batch)

In [43]:
jnp.argmax(outs[:,-1], axis=-1)

Array([0], dtype=int32)

In [44]:
jnp.argmax(labels, axis=-1)

Array([0], dtype=int32)

In [1]:
from jax import numpy as jnp

In [4]:
jnp.exp(-jnp.inf)

Array(0., dtype=float32, weak_type=True)